# Final Project: 某闯关类手游用户流失预测


## 一、案例简介

手游在当下的日常娱乐中占据着主导性地位，成为人们生活中放松身心的一种有效途径。近年来，各种类型的手游，尤其是闯关类的休闲手游，由于其对碎片化时间的利用取得了非常广泛的市场。然而在此类手游中，新用户流失是一个非常严峻的问题，有相当多的新用户在短暂尝试后会选择放弃，而如果能在用户还没有完全卸载游戏的时候针对流失可能性较大的用户施以干预（例如奖励道具、暖心短信），就可能挽回用户从而提升游戏的活跃度和公司的潜在收益，因此用户的流失预测成为一个重要且挑战性的问题。在毕业项目中我们将从真实游戏中非结构化的日志数据出发，构建用户流失预测模型，综合已有知识设计适合的算法解决实际问题。


## 二、作业说明

- 根据给出的实际数据（包括用户游玩历史，关卡特征等），预测测试集中的用户是否为流失用户（二分类）；
- 方法不限，自行进行评测，评价指标使用 AUC；
- 建议尝试使用云平台进行实验；
- 提交代码与实验报告，报告展示对数据的观察、分析、最后的解决方案以及不同尝试的对比等；
- 最终评分会参考达到的效果以及对所尝试方法的分析。


## 三、Tips

- 一个基本的思路可以是：根据游玩关卡的记录为每个用户提取特征 → 结合 label 构建表格式的数据集 → 使用不同模型训练与测试；
- 还可以借助其他模型（如循环神经网络）直接对用户历史序列建模；
- 数据量太大运行时间过长的话，可以先在一个采样的小训练集上调参；
- 集成多种模型往往能达到更优的效果；
- 可以使用各种开源工具。


## 四、实验报告


### 导入工具包


In [ ]:
%matplotlib inline
import time
import logging
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tabulate import tabulate
from sklearn.base import BaseEstimator
from imblearn.over_sampling import SMOTE
from scipy.stats import randint, uniform
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
from sklearn.neighbors import KNeighborsClassifier
from typing import Tuple, Dict, List, Optional, Union
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LassoCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import (
    SelectKBest,
    RFECV,
    SelectFromModel,
    VarianceThreshold,
    mutual_info_classif,
)
from sklearn.metrics import (
    accuracy_score,
    roc_curve,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    auc,
    f1_score,
    precision_score,
    recall_score,
)

### 环境配置


In [ ]:
plt.rcParams["font.family"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 12
sns.set_theme(style="whitegrid", font="SimHei")
N_JOBS = 1
RANDOM_SEED = 2025

# 配置日志记录器
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler()],
)  # 仅控制台输出
logger = logging.getLogger("Record")

### 定义相关的类


#### 数据加载类


In [ ]:
class DataLoader:
    """
    数据加载器类，负责加载和预处理数据集

    功能:
    - 加载所有文件中的数据
    - 返回整合后的数据字典，包括:
        ·'test_id': 测试集，用于预测，仅包含user_id
        ·'level_seq': 包含用户游玩每个关卡的记录，每一条记录是对某个关卡的一次尝试
        ·'level_meta': 每个关卡的一些统计特征，可用于表示关卡
        ·'test_label': 测试集的标签，包含user_id和label
        ·'train_label': 训练集的标签，包含user_id和label
        ·'dev_label': 验证集的标签，包含user_id和label
    """

    def __init__(self, data_dir: str = "./data/") -> None:
        """
        初始化数据加载器

        参数:
            data_dir (str): 数据目录路径，默认为'./data/'
        """

        self.data_dir: str = data_dir
        self.train_label: Optional[pd.DataFrame] = None
        self.dev_label: Optional[pd.DataFrame] = None
        self.test_id: Optional[pd.DataFrame] = None
        self.level_seq: Optional[pd.DataFrame] = None
        self.level_meta: Optional[pd.DataFrame] = None
        self.test_label: Optional[pd.DataFrame] = None

    def _get_data(self) -> None:
        """
        加载所有数据文件
        """

        logger.info("开始加载所有数据集...")
        try:
            # 加载训练集标签、验证集标签和测试集user_id
            self.train_label = pd.read_csv(self.data_dir + "train.csv", sep="\t")
            self.dev_label = pd.read_csv(self.data_dir + "dev.csv", sep="\t")
            self.test_id = pd.read_csv(self.data_dir + "test.csv", sep="\t")
            # 加载游戏记录和关卡元数据
            self.level_seq = pd.read_csv(self.data_dir + "level_seq.csv", sep="\t")
            self.level_meta = pd.read_csv(self.data_dir + "level_meta.csv", sep="\t")
            # 加载测试集真实标签
            self.test_label = pd.read_csv(self.data_dir + "Groundtruth.csv", sep=",")
            self.test_label = self.test_label.rename(
                columns={"ID": "user_id", "Label": "label"}
            )
            # 转换时间列
            self.level_seq["time"] = pd.to_datetime(
                self.level_seq["time"], errors="coerce"
            )

            logger.info("所有数据加载完成")

        except Exception as e:
            logger.error(f"加载数据时出错: {str(e)}")
            raise

    def _integrate_data(self) -> Dict[str, pd.DataFrame]:
        """
        整合所有数据集并返回一个包含所有数据集的字典

        返回:
            Dict[str, pd.DataFrame]: 包含以下键的字典:
                - 'train_label': 训练集user_id和标签
                - 'dev_label': 验证集user_id和标签
                - 'test_id': 测试集user_id
                - 'level_seq': 游戏行为序列
                - 'level_meta': 关卡元数据
                - 'test_label': 测试集user_id标签
        """

        logger.info("开始整合数据集...")

        datasets = {
            "train_label": self.train_label,
            "dev_label": self.dev_label,
            "test_id": self.test_id,
            "level_seq": self.level_seq,
            "level_meta": self.level_meta,
            "test_label": self.test_label,
        }

        logger.info("数据集整合完成")

        return datasets

    def load_all_data(self) -> Dict[str, pd.DataFrame]:
        """
        加载并返回所有数据

        返回:
            Dict[str, pd.DataFrame]: 整合后的数据集字典
        """

        self._get_data()
        return self._integrate_data()

#### 特征工程类


In [ ]:
class FeatureEngineer:
    """
    特征工程类，负责从原始数据中提取和构建特征

    功能:
    - 创建用户级聚合特征
    - 创建每日行为特征
    - 创建用户序列特征特征
    - 创建时序特征
    - 准备训练、验证和测试数据集或者准备训练集和测试集
    """

    def __init__(
        self,
        data: Dict[str, pd.DataFrame],
        if_val: bool = False,
        feature_selection_method: str = None,
        data_standardscaler: bool = True,
        use_smote: bool = True,
        smote_kwargs: Optional[Dict] = {"k_neighbors": 5, "random_state": RANDOM_SEED},
        create_user_features: bool = True,
        create_daily_features: bool = True,
        create_user_sequence_features: bool = True,
        create_sequential_features: bool = False,
    ) -> None:
        """
        初始化特征工程

        参数:
            data (Dict[str, pd.DataFrame]): 包含所有数据集的字典，包含以下key:
                - 'level_seq': 用户关卡行为序列数据
                - 'level_meta': 关卡元数据
                - 'train_label': 训练集标签
                - 'dev_label': 验证集标签
                - 'test_id': 测试集用户ID
                - 'test_label' : 测试集标签
            if_val: 是否单独地划分验证集合，默认为不划分
            feature_selection_method: 特征选择方法 ('filter', 'wrapper', 'embedded', None)，默认为None
            data_standardscaler: 数据是否标准化，默认为True
            use_smote: 合成少数类过采样的控制参数，默认为True
            smote_kwargs: 合成少数类过采样的自定义参数
            create_user_features: 是否进行基础用户特征的构建，默认为True
            create_daily_features: 是否进行每日行为特征的构建，默认为True
            create_user_sequence_features: 是否进行用户序列特征的构建，默认为True
            create_sequential_features: 是否进行时序序列特征的构建，默认为False

        属性:
            features (pd.DataFrame): 构建的特征数据集（以user_id为索引）
            train_features (pd.DataFrame): 训练集特征
            dev_features (pd.DataFrame): 验证集特征
            test_features (pd.DataFrame): 测试集特征
            train_label (pd.Series): 训练集标签
            dev_label (pd.Series): 验证集标签
            test_label (pd.Series): 测试集标签
            self.if_val (bool): 是否单独划分验证集
        """

        self.data = data
        self.features = None
        self.train_features = None
        self.dev_features = None
        self.test_features = None
        self.train_label = None
        self.dev_label = None
        self.test_label = None
        self.if_val = if_val
        self.feature_selection_method = feature_selection_method
        self.data_standardsclaer = data_standardscaler
        self.use_smote = use_smote
        self.smote_kwargs = smote_kwargs or {"random_state": RANDOM_SEED}
        self.create_user_features = create_user_features
        self.create_daily_features = create_daily_features
        self.create_user_sequence_features = create_user_sequence_features
        self.create_sequential_features = create_sequential_features

    def _create_user_features(self) -> pd.DataFrame:
        """
        为用户创建预测特征

        返回:
            合并后的特征（self.features）
        """

        logger.info("开始创建用户级特征...")
        start_time = time.time()

        try:
            # 合并游戏记录和关卡元数据
            merged_df = pd.merge(
                self.data["level_seq"],
                self.data["level_meta"],
                on="level_id",
                how="left",
            )

            # 用户级特征聚合
            agg_dict = {
                "f_success": ["sum", "mean", "count"],
                "f_duration": ["sum", "mean", "std", "min", "max", "median"],
                "f_reststep": ["mean", "min"],
                "f_help": ["sum", "mean"],
                "f_avg_duration": "mean",
                "f_avg_passrate": "mean",
                "f_avg_retrytimes": "mean",
                "time": ["min", "max", "nunique"],
            }

            user_features = merged_df.groupby("user_id").agg(agg_dict)

            # 扁平化多级列索引
            user_features.columns = [
                "_".join(col).strip() for col in user_features.columns.values
            ]

            # 计算额外特征
            user_features["session_duration"] = (
                user_features["time_max"] - user_features["time_min"]
            ).dt.total_seconds()

            # 删除多余特征
            user_features.drop(["time_max", "time_min"], axis=1, inplace=True)

            # 重命名关键特征，提高可读性
            rename_map = {
                "f_success_sum": "success_count",
                "f_success_count": "total_attempts",
                "f_success_mean": "success_rate",
                "f_help_sum": "help_used",
                "f_reststep_mean": "avg_reststep",
                "f_avg_passrate_mean": "avg_level_passrate",
            }
            user_features.rename(columns=rename_map, inplace=True)

            # 计算衍生特征
            user_features["efficiency"] = (
                user_features["success_count"] / user_features["f_duration_sum"]
            )

            self.features = user_features

            elapsed = time.time() - start_time
            logger.info(
                f"用户级特征创建完成! 耗时: {elapsed:.2f}秒，特征形状: {self.features.shape}"
            )

            return self.features

        except Exception as e:
            logger.error(f"创建用户特征时出错: {str(e)}")
            raise

    def _create_daily_features(self) -> pd.DataFrame:
        """
        创建基于每日行为的特征

        返回:
            合并后的特征（self.features）
        """

        logger.info("开始创建每日行为特征...")
        start_time = time.time()

        try:
            # 检查核心特征集是否已初始化
            if self.features is None:
                raise ValueError("请先调用_create_user_features初始化核心特征集")

            # 提取日期信息
            level_seq = self.data["level_seq"].copy()
            level_seq["date"] = level_seq["time"].dt.date  # 提取日期

            # 每日行为统计（按用户-日期分组）
            daily_stats = (
                level_seq.groupby(["user_id", "date"])
                .agg(
                    success_count=("f_success", "count"),
                    success_sum=("f_success", "sum"),
                    duration_sum=("f_duration", "sum"),
                    help_sum=("f_help", "sum"),
                )
                .reset_index()
            )

            # 按用户聚合每日统计数据（计算每日指标的分布特征）
            user_daily = daily_stats.groupby("user_id").agg(
                {
                    "success_count": ["mean", "std", "min", "max"],
                    "success_sum": ["mean", "std", "min", "max"],
                    "duration_sum": ["mean", "std", "min", "max"],
                    "help_sum": ["mean", "std", "min", "max"],
                }
            )

            # 扁平化多级列索引，添加前缀区分每日特征
            user_daily.columns = [
                "daily_" + "_".join(col).strip() for col in user_daily.columns.values
            ]

            # 合并到主特征集（左连接，保留所有用户）
            self.features = self.features.join(user_daily, how="left")

            # 填充可能的缺失值（如仅活跃1天的用户无std）
            self.features.fillna(0, inplace=True)

            elapsed = time.time() - start_time
            logger.info(
                f"每日行为特征创建完成! 耗时: {elapsed:.2f}秒，特征形状: {self.features.shape}"
            )

            return self.features

        except Exception as e:
            logger.error(f"创建每日特征时出错: {str(e)}")
            raise

    def _create_level_features(self) -> pd.DataFrame:
        """
        创建关卡级特征（用户在各关卡的行为特征），并合并到核心特征集

        返回:
            合并后的特征（self.features）
        """

        logger.info("开始创建关卡级特征...")
        start_time = time.time()

        try:
            # 检查核心特征集是否已初始化
            if self.features is None:
                raise ValueError("请先调用_create_user_features初始化核心特征集")

            # 获取关卡行为数据
            seq_df = self.data["level_seq"].copy()

            # 按用户-关卡分组计算特征
            grouped = seq_df.groupby(["user_id", "level_id"])
            level_features = grouped.agg(
                level_attempts=("f_success", "size"),  # 该关卡总尝试次数
                level_success=("f_success", "sum"),  # 该关卡成功次数
                level_retries=(
                    "f_success",
                    lambda x: x.count() - 1,
                ),  # 重试次数（总尝试-1）
                level_duration_avg=("f_duration", "mean"),  # 平均耗时
                level_duration_std=("f_duration", "std"),  # 耗时标准差
            ).reset_index()

            # 计算序列特征：是否首次尝试成功
            level_features["success_on_first_try"] = (
                level_features["level_retries"] == 0
            ).astype(int)

            # 按用户聚合关卡级特征（将多关卡特征聚合为用户级特征）
            user_level_agg = level_features.groupby("user_id").agg(
                avg_attempts_per_level=("level_attempts", "mean"),
                max_attempts_per_level=("level_attempts", "max"),
                min_attempts_per_level=("level_attempts", "min"),
                avg_retries_per_level=("level_retries", "mean"),
                max_retries_per_level=("level_retries", "max"),
                min_retries_per_level=("level_retries", "min"),
                first_try_success_rate=("success_on_first_try", "mean"),
                level_count=("level_id", "nunique"),  # 用户尝试的关卡总数
                level_id_range=(
                    "level_id",
                    lambda x: x.max() - x.min() if x.nunique() > 1 else 0,
                ),
            )

            # 合并到核心特征集
            self.features = self.features.join(user_level_agg, how="left")

            # 填充缺失值（如仅尝试1个关卡的用户无level_id_range）
            self.features.fillna(0, inplace=True)

            elapsed = time.time() - start_time
            logger.info(
                f"关卡级特征创建完成! 耗时: {elapsed:.2f}秒，特征形状: {self.features.shape}"
            )

            return self.features

        except Exception as e:
            logger.error(f"创建关卡级特征时出错: {str(e)}")
            raise

    def _create_user_sequence_features(self) -> pd.DataFrame:
        """
        创建用户级序列特征（结合关卡元数据的对比特征）

        返回:
            合并后的特征（self.features）
        """

        logger.info("开始创建用户序列特征...")
        start_time = time.time()

        try:
            # 检查核心特征集是否已初始化
            if self.features is None:
                raise ValueError("请先调用_create_user_features初始化核心特征集")

            # 确保数据按用户和时间排序
            seq_df = self.data["level_seq"][
                ["user_id", "level_id", "time", "f_success", "f_duration"]
            ].copy()
            meta_df = self.data["level_meta"][
                ["level_id", "f_avg_passrate", "f_avg_duration", "f_avg_retrytimes"]
            ].copy()

            # 先创建关卡级特征（已合并到self.features）
            self._create_level_features()

            # 合并行为数据与关卡元数据（用于计算与元数据的对比特征）
            merged_df = seq_df.merge(meta_df, on="level_id", how="left")

            # 计算用户尝试关卡的元数据统计特征
            user_meta_features = merged_df.groupby("user_id").agg(
                mean_meta_passrate=(
                    "f_avg_passrate",
                    "mean",
                ),  # 用户尝试关卡的平均预期通过率
                mean_meta_duration=(
                    "f_avg_duration",
                    "mean",
                ),  # 用户尝试关卡的平均预期耗时
                meta_retry_ratio=(
                    "f_avg_retrytimes",
                    "mean",
                ),  # 用户尝试关卡的平均预期重试率
                min_meta_passrate=(
                    "f_avg_passrate",
                    "min",
                ),  # 用户尝试关卡的最低预期通过率
                max_meta_passrate=(
                    "f_avg_passrate",
                    "max",
                ),  # 用户尝试关卡的最高预期通过率
                difficulty_range=(
                    "f_avg_passrate",
                    lambda x: x.max() - x.min() if x.nunique() > 1 else 0,
                ),  # 难度范围
            )

            # 计算用户表现与关卡预期的对比（实际成功率 - 预期平均通过率）
            user_actual_success = merged_df.groupby("user_id")["f_success"].mean()
            user_meta_features["performance_gap"] = (
                user_actual_success - user_meta_features["mean_meta_passrate"]
            )

            # 合并元数据特征到核心特征集
            self.features = self.features.join(user_meta_features, how="left")

            # 填充缺失值
            self.features.fillna(0, inplace=True)

            elapsed = time.time() - start_time
            logger.info(
                f"用户序列特征创建完成! 耗时: {elapsed:.2f}秒，特征形状: {self.features.shape}"
            )

            return self.features

        except Exception as e:
            logger.error(f"创建用户序列特征时出错: {str(e)}")
            raise

    def _create_sequential_features(self) -> pd.DataFrame:
        """
        创建时序序列特征（基于时间顺序的行为特征）

        优化点:
        - 减少apply使用
        - 优化分组操作
        - 使用向量化时间计算
        """

        logger.info("开始创建时序序列特征...")
        start_time = time.time()

        try:
            # 检查核心特征集是否已初始化
            if self.features is None:
                raise ValueError("请先调用_create_user_features初始化核心特征集")

            # 只选择必要的列减少内存占用
            seq_df = self.data["level_seq"][
                ["user_id", "level_id", "time", "f_success", "f_duration", "f_reststep"]
            ].copy()

            # 确保数据按用户和时间排序
            seq_df = seq_df.sort_values(["user_id", "time"]).reset_index(drop=True)

            # 1. 全局序列特征
            # 用户全局行为步数
            seq_df["global_step"] = seq_df.groupby("user_id").cumcount() + 1

            # 2. 按用户-关卡分组，提取关卡内的详细序列特征
            # 关卡内尝试次数
            seq_df["level_attempt_num"] = (
                seq_df.groupby(["user_id", "level_id"]).cumcount() + 1
            )

            # 时间间隔特征 - 向量化计算
            seq_df["time_diff"] = (
                seq_df.groupby(["user_id", "level_id"])["time"]
                .diff()
                .dt.total_seconds()
            )
            seq_df["time_since_first_attempt"] = seq_df.groupby(
                ["user_id", "level_id"]
            )["time"].transform(lambda x: (x - x.min()).dt.total_seconds())

            # 累计成功与尝试特征
            seq_df["cum_success_in_level"] = seq_df.groupby(["user_id", "level_id"])[
                "f_success"
            ].cumsum()
            seq_df["level_success_rate_so_far"] = (
                seq_df["cum_success_in_level"] / seq_df["level_attempt_num"]
            )

            # 耗时相关特征
            seq_df["duration_diff"] = seq_df.groupby(["user_id", "level_id"])[
                "f_duration"
            ].diff()
            seq_df["cum_avg_duration_in_level"] = (
                seq_df.groupby(["user_id", "level_id"])["f_duration"]
                .expanding()
                .mean()
                .values
            )
            seq_df["duration_ratio_to_avg"] = seq_df["f_duration"] / seq_df[
                "cum_avg_duration_in_level"
            ].replace(0, 1)

            # 剩余步骤特征
            seq_df["reststep_diff"] = seq_df.groupby(["user_id", "level_id"])[
                "f_reststep"
            ].diff()

            # 关键事件标记
            seq_df["is_first_attempt"] = (seq_df["level_attempt_num"] == 1).astype(int)

            # 连续行为模式 - 向量化计算连续失败次数
            seq_df["failure"] = (seq_df["f_success"] == 0).astype(int)
            failure_shift = seq_df["failure"] != seq_df["failure"].shift()
            seq_df["consecutive_failures"] = seq_df.groupby(
                ["user_id", "level_id", failure_shift.cumsum()]
            )["failure"].cumsum()

            # 3. 用户级全局上下文特征
            # 用户总尝试次数
            seq_df["user_total_attempts"] = seq_df.groupby("user_id")[
                "global_step"
            ].transform("max")
            # 用户累计成功次数
            seq_df["user_total_success"] = seq_df.groupby("user_id")[
                "f_success"
            ].cumsum()
            # 用户整体成功率
            seq_df["user_overall_success_rate"] = seq_df["user_total_success"] / seq_df[
                "global_step"
            ].replace(0, 1)

            # 用户已尝试的关卡数量
            seq_df["user_unique_levels_count"] = seq_df.groupby("user_id")[
                "level_id"
            ].transform("nunique")

            # 用户平均表现的动态变化
            seq_df["user_avg_duration_so_far"] = (
                seq_df.groupby("user_id")["f_duration"].expanding().mean().values
            )
            seq_df["user_avg_reststep_so_far"] = (
                seq_df.groupby("user_id")["f_reststep"].expanding().mean().values
            )

            # 当前关卡与用户历史的对比
            seq_df["duration_vs_user_avg"] = seq_df["f_duration"] / seq_df[
                "user_avg_duration_so_far"
            ].replace(0, 1)
            seq_df["reststep_vs_user_avg"] = seq_df["f_reststep"] / seq_df[
                "user_avg_reststep_so_far"
            ].replace(0, 1)

            # 聚合序列特征 - 按用户分组，提取统计量
            # 取每个用户的最后一条记录
            last_record = seq_df.groupby("user_id").last()

            # 构建时序特征集
            sequential_features = pd.DataFrame(
                {
                    "seq_global_step_max": last_record["global_step"],
                    "seq_level_attempt_num_mean": seq_df.groupby("user_id")[
                        "level_attempt_num"
                    ].mean(),
                    "seq_level_attempt_num_max": seq_df.groupby("user_id")[
                        "level_attempt_num"
                    ].max(),
                    "seq_time_since_prev_attempt_mean": seq_df.groupby("user_id")[
                        "time_diff"
                    ].mean(),
                    "seq_time_since_prev_attempt_std": seq_df.groupby("user_id")[
                        "time_diff"
                    ].std(),
                    "seq_time_since_first_attempt_max": last_record[
                        "time_since_first_attempt"
                    ],
                    "seq_level_success_rate_so_far_last": last_record[
                        "level_success_rate_so_far"
                    ],
                    "seq_duration_change_from_prev_mean": seq_df.groupby("user_id")[
                        "duration_diff"
                    ].mean(),
                    "seq_duration_change_from_prev_std": seq_df.groupby("user_id")[
                        "duration_diff"
                    ].std(),
                    "seq_cum_avg_duration_in_level_last": last_record[
                        "cum_avg_duration_in_level"
                    ],
                    "seq_duration_ratio_to_avg_mean": seq_df.groupby("user_id")[
                        "duration_ratio_to_avg"
                    ].mean(),
                    "seq_duration_ratio_to_avg_std": seq_df.groupby("user_id")[
                        "duration_ratio_to_avg"
                    ].std(),
                    "seq_reststep_change_from_prev_mean": seq_df.groupby("user_id")[
                        "reststep_diff"
                    ].mean(),
                    "seq_reststep_change_from_prev_std": seq_df.groupby("user_id")[
                        "reststep_diff"
                    ].std(),
                    "seq_is_first_attempt_sum": seq_df.groupby("user_id")[
                        "is_first_attempt"
                    ].sum(),
                    "seq_consecutive_failures_mean": seq_df.groupby("user_id")[
                        "consecutive_failures"
                    ].mean(),
                    "seq_consecutive_failures_max": seq_df.groupby("user_id")[
                        "consecutive_failures"
                    ].max(),
                    "seq_user_total_attempts_max": last_record["user_total_attempts"],
                    "seq_user_total_success_max": last_record["user_total_success"],
                    "seq_user_overall_success_rate_last": last_record[
                        "user_overall_success_rate"
                    ],
                    "seq_user_unique_levels_count_max": last_record[
                        "user_unique_levels_count"
                    ],
                    "seq_user_avg_duration_so_far_last": last_record[
                        "user_avg_duration_so_far"
                    ],
                    "seq_user_avg_reststep_so_far_last": last_record[
                        "user_avg_reststep_so_far"
                    ],
                    "seq_duration_vs_user_avg_mean": seq_df.groupby("user_id")[
                        "duration_vs_user_avg"
                    ].mean(),
                    "seq_duration_vs_user_avg_std": seq_df.groupby("user_id")[
                        "duration_vs_user_avg"
                    ].std(),
                    "seq_reststep_vs_user_avg_mean": seq_df.groupby("user_id")[
                        "reststep_vs_user_avg"
                    ].mean(),
                    "seq_reststep_vs_user_avg_std": seq_df.groupby("user_id")[
                        "reststep_vs_user_avg"
                    ].std(),
                }
            )

            # 合并到主特征集
            self.features = self.features.join(sequential_features, how="left")

            # 填充时序特征中的缺失值
            self.features.fillna(0, inplace=True)

            elapsed = time.time() - start_time
            logger.info(
                f"时序序列特征创建完成! 耗时: {elapsed:.2f}秒，特征形状: {self.features.shape}"
            )

            return self.features

        except Exception as e:
            logger.error(f"创建序列特征时出错: {str(e)}")
            raise

    def create_all_features(
        self,
        create_user_features: Optional[bool] = None,
        create_daily_features: Optional[bool] = None,
        create_user_sequence_features: Optional[bool] = None,
        create_sequential_features: Optional[bool] = None,
    ) -> pd.DataFrame:
        """
        按顺序创建所有特征，确保特征正确累积

        参数:
            create_user_features: 是否进行基础用户特征的构建，默认为是
            create_daily_features: 是否进行每日行为特征的构建，默认为是
            create_user_sequence_features: 是否进行用户序列特征的构建，默认为是
            create_sequential_features: 是否进行时序序列特征的构建，默认为否

        返回:
            完整的特征集（self.features）
        """

        logger.info("开始创建所有特征...")
        start_time = time.time()

        # 如果未指定参数，使用类初始化时的默认值
        if create_user_features is None:
            create_user_features = self.create_user_features
        if create_daily_features is None:
            create_daily_features = self.create_daily_features
        if create_user_sequence_features is None:
            create_user_sequence_features = self.create_user_sequence_features
        if create_sequential_features is None:
            create_sequential_features = self.create_sequential_features

        # 按依赖顺序创建特征
        if create_user_features == True:
            self._create_user_features()  # 基础用户特征（初始化self.features）
        if create_daily_features == True:
            self._create_daily_features()  # 每日行为特征（依赖基础用户特征）
        if create_user_sequence_features == True:
            self._create_user_sequence_features()  # 用户序列特征（依赖基础用户特征，内部调用关卡级特征）
        if create_sequential_features == True:
            self._create_sequential_features()  # 时序序列特征（依赖基础用户特征）

        elapsed = time.time() - start_time
        logger.info(
            f"所有特征创建完成! 总耗时: {elapsed:.2f}秒，最终特征形状: {self.features.shape}"
        )

        return self.features

    def apply_feature_selection(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_dev: pd.DataFrame = None,
        X_test: Optional[pd.DataFrame] = None,
        method: Optional[str] = None,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """
        应用特征选择方法并返回筛选后的数据集

        参数:
            method: 特征选择方法 ('filter', 'wrapper', 'embedded', None)
        返回:
            筛选后的特征集 (X_train_sel, X_dev_sel, X_test_sel)
        """

        if method is None:
            if X_dev != None:
                return X_train, X_dev, X_test
            else:
                return X_train, X_test

        logger.info(f"开始应用特征选择: {method}方法...")
        start_time = time.time()

        original_features = X_train.shape[1]

        try:
            # 过滤式特征选择（Filter）
            if method == "filter":  # 先移除低方差特征（阈值=0.01）
                var_selector = VarianceThreshold(threshold=0.01)
                X_train_filtered = var_selector.fit_transform(X_train)
                if X_dev != None:
                    X_dev_filtered = var_selector.transform(X_dev)
                X_test_filtered = var_selector.transform(X_test)

                # 再选择Top 50互信息特征
                k = (
                    min(50, X_train_filtered.shape[1])
                    if X_train_filtered.shape[1] > 50
                    else "all"
                )
                selector = SelectKBest(score_func=mutual_info_classif, k=k)
                X_train_sel = selector.fit_transform(X_train_filtered, y_train)
                if X_dev != None:
                    X_dev_sel = selector.transform(X_dev_filtered)
                X_test_sel = selector.transform(X_test_filtered)
                selected_mask = selector.get_support()

            # 包裹式特征选择（Wrapper）
            elif method == "wrapper":
                # 使用带交叉验证的递归特征消除
                estimator = RandomForestClassifier(
                    n_estimators=100, random_state=RANDOM_SEED
                )

                selector = RFECV(
                    estimator=estimator,
                    step=0.05,  # 每次迭代移除5%的特征
                    cv=5,
                    scoring="accuracy",
                    min_features_to_select=10,
                )

                X_train_sel = selector.fit_transform(X_train, y_train)
                if X_dev != None:
                    X_dev_sel = selector.transform(X_dev)
                X_test_sel = selector.transform(X_test)
                selected_mask = selector.support_

            # 嵌入式特征选择（Embedded）
            elif method == "embedded":
                # 使用Lasso回归选择特征
                selector = SelectFromModel(
                    LassoCV(cv=5, random_state=RANDOM_SEED), threshold="1.25*median"
                )

                X_train_sel = selector.fit_transform(X_train, y_train)
                if X_dev != None:
                    X_dev_sel = selector.transform(X_dev)
                X_test_sel = selector.transform(X_test)
                selected_mask = selector.get_support()

            else:
                raise ValueError(f"不支持的feature_selection_method: {method}")

            # 记录被选中的特征名
            selected_features = X_train.columns[selected_mask].tolist()

            logger.info(
                f"特征选择完成！原始特征数: {original_features} → 筛选后: {len(selected_features)}"
            )
            logger.debug(f"选中特征列表: {selected_features}")

            elapsed = time.time() - start_time
            logger.info(f"特征选择耗时: {elapsed:.2f}秒")

            if X_dev != None:
                return X_train_sel, X_dev_sel, X_test_sel
            else:
                return X_train_sel, X_test_sel

        except Exception as e:
            logger.error(f"特征选择失败: {str(e)}")
            raise

    def prepare_datasets(self, feature_selection_method: Optional[str] = None) -> Tuple:
        """
        准备训练、验证和测试数据集，支持SMOTE过采样

        参数:
            feature_selection_method: 特征选择方法 ('filter', 'wrapper', 'embedded', None)

        返回:
            Tuple: (X_train, y_train, X_dev, y_dev, X_test, y_test) 或 (X_full_train, y_full_train, X_test, y_test)
        """

        logger.info("开始准备数据集...")
        start_time = time.time()

        try:
            # 确保特征已创建
            if self.features is None:
                self.create_all_features()

            # 合并特征和标签
            train_data = pd.merge(
                self.data["train_label"],
                self.features,
                left_on="user_id",
                right_index=True,
                how="left",
            )

            dev_data = pd.merge(
                self.data["dev_label"],
                self.features,
                left_on="user_id",
                right_index=True,
                how="left",
            )

            test_data = pd.merge(
                self.data["test_label"],
                self.features,
                left_on="user_id",
                right_index=True,
                how="left",
            )

            # 填充可能存在的缺失值
            train_data.fillna(0, inplace=True)
            dev_data.fillna(0, inplace=True)
            test_data.fillna(0, inplace=True)

            # 分离特征和标签
            if self.if_val:
                X_train = train_data.drop(columns=["user_id", "label"])
                y_train = train_data["label"]
                X_dev = dev_data.drop(columns=["user_id", "label"])
                y_dev = dev_data["label"]
                X_test = test_data.drop(columns=["user_id", "label"])
                y_test = test_data["label"]

                # 特征选择
                if feature_selection_method:
                    X_train_sel, X_dev_sel, X_test_sel = self.apply_feature_selection(
                        X_train, y_train, X_dev, X_test, method=feature_selection_method
                    )
                else:
                    X_train_sel, X_dev_sel, X_test_sel = X_train, X_dev, X_test

                # 标准化处理
                if self.data_standardsclaer:
                    logger.info("开始对数据集进行标准化...")
                    scaler = StandardScaler()
                    X_train_scaled = scaler.fit_transform(X_train_sel)
                    X_train_scaled = pd.DataFrame(
                        X_train_scaled,
                        columns=X_train_sel.columns,
                        index=X_train_sel.index,
                    )
                    X_dev_scaled = scaler.transform(X_dev_sel)
                    X_dev_scaled = pd.DataFrame(
                        X_dev_scaled, columns=X_dev_sel.columns, index=X_dev_sel.index
                    )
                    X_test_scaled = scaler.transform(X_test_sel)
                    X_test_scaled = pd.DataFrame(
                        X_test_scaled,
                        columns=X_test_sel.columns,
                        index=X_test_sel.index,
                    )
                    logger.info("数据集标准化完成")
                else:
                    X_train_scaled, X_dev_scaled, X_test_scaled = (
                        X_train_sel,
                        X_dev_sel,
                        X_test_sel,
                    )

                # SMOTE过采样处理
                if self.use_smote:
                    logger.info("开始应用SMOTE过采样...")
                    try:
                        # 记录原始分布
                        orig_counts = y_train.value_counts()
                        logger.info(
                            f"过采样前类别分布: 0类={orig_counts[0]}, 1类={orig_counts[1]}"
                        )
                        # 应用SMOTE
                        smote = SMOTE(**self.smote_kwargs)
                        X_resampled, y_resampled = smote.fit_resample(
                            X_train_scaled, y_train
                        )
                        # 更新数据集
                        X_train_scaled, y_train = X_resampled, y_resampled
                        self.train_features = X_train_scaled
                        self.train_label = y_train
                        # 记录采样后分布
                        resampled_counts = pd.Series(y_resampled).value_counts()
                        number_rise = (
                            resampled_counts[0]
                            + resampled_counts[1]
                            - orig_counts[0]
                            - orig_counts[1]
                        )
                        logger.info(
                            f"过采样后类别分布: 0类={resampled_counts[0]}, 1类={resampled_counts[1]}"
                        )
                        logger.info(f"训练集样本量增加: {number_rise}")

                    except Exception as e:
                        logger.error(f"SMOTE过采样失败: {str(e)}")
                        raise

                self.train_features = X_train_scaled
                self.train_label = y_train
                self.dev_features = X_dev_scaled
                self.dev_label = y_dev
                self.test_features = X_test_scaled
                self.test_label = y_test

                elapsed = time.time() - start_time
                logger.info(f"数据集准备完成! 耗时: {elapsed:.2f}秒")
                logger.info(
                    f"训练集形状: {X_train_scaled.shape}, 验证集形状: {X_dev_scaled.shape}, 测试集形状: {X_test_scaled.shape}"
                )

                return (
                    X_train_scaled,
                    y_train,
                    X_dev_scaled,
                    y_dev,
                    X_test_scaled,
                    y_test,
                )

            else:
                # 合并训练集和验证集
                full_train_data = pd.concat(
                    [train_data, dev_data], axis=0, ignore_index=True
                )

                # 分离特征和标签
                X_full_train = full_train_data.drop(columns=["user_id", "label"])
                y_full_train = full_train_data["label"]
                X_test = test_data.drop(columns=["user_id", "label"])
                y_test = test_data["label"]

                # 特征选择
                if feature_selection_method:
                    X_full_train_sel, X_test_sel = self.apply_feature_selection(
                        X_full_train,
                        y_full_train,
                        None,
                        X_test,
                        method=feature_selection_method,
                    )
                else:
                    X_full_train_sel, X_test_sel = X_full_train, X_test

                # 标准化处理
                if self.data_standardsclaer:
                    logger.info("开始对数据集进行标准化...")
                    scaler = StandardScaler()
                    X_full_train_scaled = scaler.fit_transform(X_full_train_sel)
                    X_full_train_scaled = pd.DataFrame(
                        X_full_train_scaled,
                        columns=X_full_train_sel.columns,
                        index=X_full_train_sel.index,
                    )
                    X_test_scaled = scaler.transform(X_test_sel)
                    X_test_scaled = pd.DataFrame(
                        X_test_scaled,
                        columns=X_test_sel.columns,
                        index=X_test_sel.index,
                    )
                    logger.info("数据集标准化完成")
                else:
                    X_full_train_scaled, X_test_scaled = X_full_train_sel, X_test_sel

                # SMOTE过采样处理
                if self.use_smote:
                    logger.info("开始应用SMOTE过采样...")
                    try:
                        # 记录原始分布
                        orig_counts = y_full_train.value_counts()
                        logger.info(
                            f"过采样前类别分布: 0类={orig_counts[0]}, 1类={orig_counts[1]}"
                        )
                        # 应用SMOTE
                        smote = SMOTE(**self.smote_kwargs)
                        X_resampled, y_resampled = smote.fit_resample(
                            X_full_train_scaled, y_full_train
                        )
                        # 更新数据集
                        X_full_train_scaled, y_full_train = X_resampled, y_resampled
                        self.train_features = X_full_train_scaled
                        self.train_label = y_full_train
                        # 记录采样后分布
                        resampled_counts = pd.Series(y_resampled).value_counts()
                        number_rise = (
                            resampled_counts[0]
                            + resampled_counts[1]
                            - orig_counts[0]
                            - orig_counts[1]
                        )
                        logger.info(
                            f"过采样后类别分布: 0类={resampled_counts[0]}, 1类={resampled_counts[1]}"
                        )
                        logger.info(f"训练集样本量增加: {number_rise}")

                    except Exception as e:
                        logger.error(f"SMOTE过采样失败: {str(e)}")
                        raise

                self.train_features = X_full_train_scaled
                self.train_label = y_full_train
                self.dev_features = None
                self.dev_label = None
                self.test_features = X_test_scaled
                self.test_label = y_test

                elapsed = time.time() - start_time
                logger.info(f"数据集准备完成! 耗时: {elapsed:.2f}秒")
                logger.info(
                    f"训练集形状: {X_full_train_scaled.shape}, 测试集形状: {X_test_scaled.shape}"
                )

                return X_full_train_scaled, y_full_train, X_test_scaled, y_test

        except Exception as e:
            logger.error(f"准备数据集时出错: {str(e)}")
            raise

#### 模型构建与优化类


In [ ]:
class ModelTrainer:
    """
    模型训练和评估类，负责训练多个模型并测试其性能
    """

    def __init__(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_test: pd.DataFrame,
        y_test: pd.Series,
    ) -> None:
        """
        初始化模型训练器

        参数:
                X_train (pd.DataFrame): 训练特征
                y_train (pd.Series): 训练标签
                X_test (pd.DataFrame): 测试特征
                y_test (pd.Series): 测试标签
        """

        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test
        self.models = {}
        self.results = []

    def train_model(
        self,
        model: BaseEstimator,
        model_name: str = None,
        param_dist: Optional[Dict] = None,
        n_iter: int = 100,
    ) -> None:
        """
        训练单个模型并进行超参数调优

        参数:
                model (BaseEstimator): 要训练的模型对象
                model_name (str): 模型名称
                param_dist (Dict): 超参数分布，用于随机搜索
                n_iter (int): 随机搜索迭代次数，默认为100次
        """

        logger.info(f"开始训练 {model_name} 模型...")
        start_time = time.time()

        try:
            # 忽略所有Python内置警告
            warnings.filterwarnings("ignore", category=Warning)

            # 忽略特定Scikit-learn警告
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            warnings.filterwarnings("ignore", category=UserWarning)
            warnings.filterwarnings("ignore", category=FutureWarning)
            warnings.filterwarnings("ignore", category=DeprecationWarning)

            # LightGBM专属警告处理
            if "LightGBM" in model_name:
                try:
                    warnings.filterwarnings(
                        "ignore", category=lgb.basic.LightGBMDeprecationWarning
                    )
                except:
                    pass

            if param_dist is not None:
                random_search = RandomizedSearchCV(  # 使用随机网格搜索进行超参数调优
                    estimator=model,
                    param_distributions=param_dist,
                    n_iter=n_iter,
                    cv=5,
                    scoring="roc_auc",
                    n_jobs=N_JOBS,
                    random_state=RANDOM_SEED,
                    verbose=1,
                )

                random_search.fit(self.X_train, self.y_train)

                # 使用最佳参数重新训练模型
                best_params = random_search.best_params_
                model = random_search.best_estimator_
                logger.info(f"{model_name} 最佳参数: {best_params}")

            else:
                model.fit(self.X_train, self.y_train)

            # 保存模型
            self.models[model_name] = model

            # 评估模型
            self.evaluate_model(model_name)

            elapsed = time.time() - start_time
            logger.info(f"{model_name} 模型训练完成! 耗时: {elapsed:.2f}秒")

        except Exception as e:
            logger.error(f"训练 {model_name} 模型时出错: {str(e)}")
            raise

    def evaluate_model(self, model_name: str) -> Dict:
        """
        评估模型，并绘制AUC曲线并生成评价指标表格

        参数:
                model_name (str): 要评估的模型名称

        返回:
                Dict: 包含评估结果的字典
        """

        model = self.models[model_name]

        # 对训练集和测试集进行预测
        train_preds = model.predict_proba(self.X_train)[:, 1]
        test_preds = model.predict_proba(self.X_test)[:, 1]

        # 计算评估指标
        train_auc = roc_auc_score(self.y_train, train_preds)
        test_auc = roc_auc_score(self.y_test, test_preds)
        test_report = classification_report(
            self.y_test,
            (test_preds > 0.5).astype(int),
            output_dict=True,
            zero_division=0,
        )

        # 存储结果
        result = {
            "model": model_name,
            "train_auc": train_auc,
            "test_auc": test_auc,
            "test_precision": test_report["weighted avg"]["precision"],
            "test_recall": test_report["weighted avg"]["recall"],
            "test_f1": test_report["weighted avg"]["f1-score"],
        }
        self.results.append(result)

        # 绘制ROC曲线
        fpr_test, tpr_test, _ = roc_curve(self.y_test, test_preds)
        plt.figure(figsize=(9, 7))
        plt.plot(
            fpr_test,
            tpr_test,
            color="darkorange",
            lw=2,
            label=f"{model_name} ROC曲线 (测试集-AUC = {test_auc:.4f})",
        )
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
        plt.xlim([0.0, 1.01])
        plt.ylim([0.0, 1.01])
        plt.xlabel("假正例率(FPR)", fontsize=10, color="blue")
        plt.ylabel("真正例率(TPR)", fontsize=10, color="green")
        plt.title(f"{model_name} - 测试集ROC曲线与AUC", fontsize=12)
        plt.legend(loc="lower right")
        plt.show()

        # 打印分类报告
        logger.info(f"\n{model_name} 模型分类报告:")
        logger.info(f"训练集 AUC: {train_auc:.4f}")
        logger.info(f"测试集 AUC: {test_auc:.4f}")
        logger.info("测试集详细指标:")
        logger.info(f"精确率: {test_report['weighted avg']['precision']:.4f}")
        logger.info(f"召回率: {test_report['weighted avg']['recall']:.4f}")
        logger.info(f"F1分数: {test_report['weighted avg']['f1-score']:.4f}")

        return result

    def compare_models(self) -> None:
        """
        比较所有模型的性能并生成表格
        """

        if not self.results:
            logger.warning("没有可比较的模型结果")
            return

        headers = [
            "模型",
            "训练集-AUC",
            "测试集-AUC",
            "测试集-查准率",
            "测试集-查全率",
            "测试集-F1分数",
        ]
        table_data = []

        for result in self.results:
            table_data.append(
                [
                    result["model"],
                    f"{result['train_auc']:.4f}",
                    f"{result['test_auc']:.4f}",
                    f"{result['test_precision']:.4f}",
                    f"{result['test_recall']:.4f}",
                    f"{result['test_f1']:.4f}",
                ]
            )

        print("\n模型性能比较:")
        print(
            tabulate(
                table_data,
                headers=headers,
                tablefmt="grid",
                numalign="center",
                stralign="center",
            )
        )
        logger.info("模型性能比较完成!")

#### 模型融合类


In [ ]:
class ModelEnsemble:
    """
    模型融合类，负责组合多个模型的预测结果并评估融合性能

    属性:
        models (Dict[str, BaseEstimator]): 存储待融合的模型字典，键为模型名称，值为模型对象
    """

    def __init__(
        self,
        models: Dict[str, BaseEstimator],
        method: str = "average",
        weights: Optional[List[float]] = None,
    ) -> None:
        """
        初始化模型融合器

        参数:
            models (Dict[str, BaseEstimator]): 待融合的模型字典
                键: 模型名称 (str)
                值: 实现predict_proba方法的模型对象 (BaseEstimator)
            method (str): 融合方法，支持:
                'average' - 等权重平均（默认）
                'weighted' - 加权平均
            weights (List[float], optional): 当method='weighted'时使用的权重列表，
                长度应与self.models的数量一致
        """

        self.models = models
        self.method = method
        self.weights = weights

    def ensemble_predict(
        self, X: pd.DataFrame, method: str = None, weights: Optional[List[float]] = None
    ) -> np.ndarray:
        """
        使用融合方法组合多个模型的预测结果

        参数:
            X (pd.DataFrame): 待预测的特征数据集，形状为(n_samples, n_features)
            method (str): 融合方法，支持:
                'average' - 等权重平均（默认）
                'weighted' - 加权平均
            weights (List[float], optional): 当method='weighted'时使用的权重列表，
                长度应与self.models的数量一致

        返回:
            np.ndarray: 融合后的预测概率，形状为(n_samples,)
        """

        if method == None:
            method = self.method
        if method == "weighted" and weights == None:
            weights = self.weights

        all_preds = []

        # 收集所有模型的预测结果
        for _, model in self.models.items():
            preds = model.predict_proba(X)[:, 1]
            all_preds.append(preds)

        all_preds = np.array(all_preds)

        # 应用融合方法
        if method == "average":
            return np.mean(all_preds, axis=0)
        elif method == "weighted":
            if weights is None:
                raise ValueError("加权融合需要提供weights参数")
            if len(weights) != len(self.models):
                raise ValueError(
                    f"权重数量({len(weights)})与模型数量({len(self.models)})不匹配"
                )
            return np.average(all_preds, axis=0, weights=weights)
        else:
            raise ValueError(f"不支持的融合方法: {method}")

    def evaluate_ensemble(
        self,
        X_train: pd.DataFrame,
        y_train: pd.Series,
        X_test: pd.DataFrame,
        y_test: pd.Series,
        method: str = "average",
    ) -> Dict[str, float]:
        """
        评估融合模型性能

        参数:
            X_test (pd.DataFrame): 训练集特征数据，形状为(n_samples, n_features)
            y_test (pd.Series): 训练集真实标签，形状为(n_samples,)
            X_test (pd.DataFrame): 测试集特征数据，形状为(n_samples, n_features)
            y_test (pd.Series): 测试集真实标签，形状为(n_samples,)
            method (str): 融合方法，支持 'average' 或 'weighted'

        返回:
            Dict[str, float]: 包含评估指标的字典，包括:
                'method': 使用的融合方法
                'auc_train': 训练集ROC曲线下面积
                'auc_test': 测试集ROC曲线下面积
                'precision': 加权平均精确率
                'recall': 加权平均召回率
                'f1': 加权平均F1分数
        """

        # 获取融合预测结果
        ensemble_preds_train = self.ensemble_predict(X_train, method)
        ensemble_preds_test = self.ensemble_predict(X_test, method)

        # 二值化预测结果
        ensemble_preds_binary_test = (ensemble_preds_test > 0.5).astype(int)

        # 计算评估指标
        auc_score_train = roc_auc_score(y_train, ensemble_preds_train)
        auc_score_test = roc_auc_score(y_test, ensemble_preds_test)
        report = classification_report(
            y_test, ensemble_preds_binary_test, output_dict=True, zero_division=0
        )

        return {
            "method": method,
            "auc_train": auc_score_train,
            "auc_test": auc_score_test,
            "precision": report["weighted avg"]["precision"],
            "recall": report["weighted avg"]["recall"],
            "f1": report["weighted avg"]["f1-score"],
        }

#### 结果可视化类


In [ ]:
class ResultVisualizer:
    """
    结果可视化类，负责展示模型性能和分析结果

    功能:
    - 绘制ROC曲线
    - 绘制特征重要性图
    - 绘制混淆矩阵
    - 绘制多模型ROC曲线对比
    """

    @staticmethod
    def plot_roc_curve(
        y_true: np.ndarray, y_score: np.ndarray, model_name: str
    ) -> None:
        """
        绘制ROC曲线

        参数:
            y_true (np.ndarray): 真实标签
            y_score (np.ndarray): 预测概率
            model_name (str): 模型名称
        """

        fpr, tpr, _ = roc_curve(y_true, y_score, drop_intermediate=False)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(9, 7))
        plt.plot(
            fpr, tpr, color="darkorange", lw=2, label=f"ROC曲线 (AUC = {roc_auc:.4f})"
        )
        plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
        plt.xlim([0.0, 1.01])
        plt.ylim([0.0, 1.01])
        plt.xlabel("假正例率(FPR)", fontsize=10, color="blue")
        plt.ylabel("真正例率(TPR)", fontsize=10, color="green")
        plt.title(f"{model_name} - 测试集ROC曲线与AUC", fontsize=12)
        plt.legend(loc="lower right")
        plt.show()

    @staticmethod
    def plot_feature_importance(
        model: Union[BaseEstimator, lgb.Booster, xgb.Booster],
        feature_names: List[str],
        model_name: str,
        top_n: int = 10,
    ) -> None:
        """
        绘制特征重要性图

        参数:
            model: 模型对象
            feature_names (List[str]): 特征名称列表
            model_name (str): 模型名称
            top_n (int): 显示前N个重要特征，默认为10
        """

        # 根据模型类型获取特征重要性
        if hasattr(model, "feature_importances_"):
            importances = model.feature_importances_
        elif hasattr(model, "coef_"):
            importances = np.abs(model.coef_[0])
        elif isinstance(model, lgb.Booster):
            importances = model.feature_importance(importance_type="gain")
        elif hasattr(model, "get_score"):
            importance_dict = model.get_score(importance_type="gain")
            importances = np.array([importance_dict.get(f, 0) for f in feature_names])
        else:
            logger.warning(f"无法获取 {model_name} 模型的特征重要性")
            return

        # 排序特征重要性
        indices = np.argsort(importances)[::-1][:top_n]
        sorted_importances = importances[indices]
        sorted_features = [feature_names[i] for i in indices]

        # 绘制条形图
        plt.figure(figsize=(9, 7))
        plt.barh(range(top_n), sorted_importances, align="center", color="skyblue")
        plt.xlabel("特征重要性", fontsize=10, color="blue")
        plt.yticks(range(top_n), sorted_features, fontsize=10, color="green")
        plt.title(f"{model_name} - 特征重要性 (Top {top_n})", fontsize=12)
        plt.gca().invert_yaxis()  # 重要性从高到低显示
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_confusion_matrix(
        y_true: np.ndarray, y_pred: np.ndarray, model_name: str
    ) -> None:
        """
        绘制混淆矩阵

        参数:
            y_true (np.ndarray): 真实标签
            y_pred (np.ndarray): 预测标签
            model_name (str): 模型名称
        """

        cm = confusion_matrix(y_true, y_pred)

        plt.figure(figsize=(9, 7))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["未流失", "流失"],
            yticklabels=["未流失", "流失"],
        )
        plt.xlabel("预测标签", fontsize=10, color="blue")
        plt.ylabel("真实标签", fontsize=10, color="green")
        plt.title(f"{model_name} - 混淆矩阵", fontsize=15)
        plt.show()

    @staticmethod
    def plot_multi_roc(
        y_true: np.ndarray, predictions: List[np.ndarray], model_names: List[str]
    ) -> None:
        """
        绘制多个分类模型的ROC曲线对比图

        参数:
            y_true (np.ndarray): 真实标签数组，形状为(n_samples,)
            predictions (List[np.ndarray]): 模型预测概率列表
            model_names (List[str]): 模型名称列表
        """

        plt.figure(figsize=(9, 7))

        for i, (y_pred, name) in enumerate(zip(predictions, model_names)):
            fpr, tpr, _ = roc_curve(y_true, y_pred)
            auc_score = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{name} (测试集-AUC={auc_score:.4f})")

        plt.plot([0, 1], [0, 1], "k--", label="随机猜测(AUC=0.5)")
        plt.xlim([0.0, 1.01])
        plt.ylim([0.0, 1.01])
        plt.xlabel("假正例率 (FPR)", fontsize=10, color="blue")
        plt.ylabel("真正例率 (TPR)", fontsize=10, color="green")
        plt.title("多模型ROC曲线对比", fontsize=12, color="purple")
        plt.legend(loc="lower right")
        plt.show()

    @staticmethod
    def plot_performance_comparison(results: pd.DataFrame) -> None:
        """
        绘制模型性能对比柱状图

        参数:
            results (pd.DataFrame): 包含模型性能的DataFrame
        """

        # 准备数据
        models = results["模型"].tolist()
        metrics = ["AUC", "准确率", "查准率", "查全率", "F1分数"]
        n_models = len(models)
        n_metrics = len(metrics)
        # 设置图形大小
        plt.figure(figsize=(14, 7))
        # 创建位置索引
        x = np.arange(n_models)
        # 颜色方案
        colors = [
            "#1f77b4",  # 蓝色
            "#ff7f0e",  # 橙色
            "#2ca02c",  # 绿色
            "#d62728",  # 红色
            "#9467bd",  # 紫色
        ]

        # 设置柱状图宽度和间距
        bar_width = 0.12
        spacing = 0.02

        # 绘制每种指标的柱状图并添加数值标签
        for i, metric in enumerate(metrics):
            # 计算每个柱子的位置
            positions = x + i * (bar_width + spacing)
            # 获取当前指标的数值
            values = results[metric].tolist()
            # 绘制柱状图
            bars = plt.bar(
                positions,
                values,
                bar_width,
                color=colors[i],
                label=metric,
                # edgecolor = 'black' # 添加边框
            )
            # 在每个柱子上方添加数值标签
            # for bar in bars:
            #    height = bar.get_height()
            #    plt.text(
            #        bar.get_x() + bar.get_width() / 2.0,
            #        height + 0.01, # 在柱子顶部上方添加标签
            #        f'{height:.4f}',
            #        ha = 'center',
            #        va = 'bottom',
            #        fontsize = 6
            #    )

        # 设置图表标题和标签
        plt.xlabel("模型", fontsize=12, color="blue")
        plt.ylabel("得分", fontsize=12, color="green")
        plt.title("模型性能对比", fontsize=15)
        # 设置x轴刻度和标签
        plt.xticks(x + (bar_width * (n_metrics - 1) / 2), models, rotation=90)
        # 设置y轴范围
        plt.ylim(0.4, 0.85)
        # 将图例放在图表外部（右上角）
        plt.legend(
            loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True, title="性能指标"
        )
        # 添加网格线（水平方向）
        plt.grid(axis="y", alpha=0.7)
        # 调整布局，确保图例不会被裁剪
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        # 显示图表
        plt.show()

### 运行与结果展示


In [ ]:
# 初始化数据加载器
data_loader = DataLoader()
datasets = data_loader.load_all_data()

In [ ]:
# 特征工程
feature_engineer = FeatureEngineer(data=datasets)
all_features = feature_engineer.create_all_features()

In [ ]:
# 查看所有特征
all_features

共有 13589 个样本，51 个特征。


In [ ]:
# 准备数据集
X_train, y_train, X_test, y_test = feature_engineer.prepare_datasets()

In [ ]:
# 初始化模型训练器
trainer = ModelTrainer(X_train, y_train, X_test, y_test)

In [ ]:
# 定义模型的超参数空间
models = {
    "决策树": (
        DecisionTreeClassifier(random_state=RANDOM_SEED),
        {
            "max_depth": [None] + list(range(5, 21, 5)),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "criterion": ["gini", "entropy"],
        },
    ),
    "逻辑回归": (
        LogisticRegression(max_iter=3000, random_state=RANDOM_SEED),
        {"C": uniform(0.1, 10), "penalty": ["l1", "l2"], "solver": ["saga"]},
    ),
    "朴素贝叶斯": (GaussianNB(), {"var_smoothing": uniform(1e-10, 1e-5)}),
    "K近邻": (
        KNeighborsClassifier(),
        {
            "n_neighbors": randint(3, 15),
            "weights": ["uniform", "distance"],
            "p": [1, 2],
            "leaf_size": randint(20, 50),
        },
    ),
    "支持向量机": (
        SVC(probability=True, random_state=RANDOM_SEED),
        {
            "C": uniform(0.1, 10),
            "kernel": ["linear", "rbf", "sigmoid"],
            "gamma": [float(x) for x in np.logspace(-3, 1, 5)] + ["scale", "auto"],
        },
    ),
    "随机森林": (
        RandomForestClassifier(random_state=RANDOM_SEED),
        {
            "n_estimators": randint(100, 300),
            "max_depth": [None] + list(range(10, 31, 5)),
            "min_samples_split": randint(2, 10),
            "max_features": ["sqrt", "log2"],
        },
    ),
    "梯度提升树": (
        GradientBoostingClassifier(random_state=RANDOM_SEED),
        {
            "learning_rate": uniform(0.01, 0.2),
            "n_estimators": randint(100, 300),
            "max_depth": randint(3, 8),
            "subsample": uniform(0.7, 0.3),
        },
    ),
    "XGBoost": (
        xgb.XGBClassifier(verbosity=0, random_state=RANDOM_SEED),
        {
            "learning_rate": uniform(0.01, 0.3),
            "max_depth": randint(3, 10),
            "n_estimators": randint(100, 300),
            "subsample": uniform(0.7, 0.3),
            "colsample_bytree": uniform(0.7, 0.3),
        },
    ),
    "LightGBM": (
        lgb.LGBMClassifier(verbosity=-1, random_state=RANDOM_SEED),
        {
            "num_leaves": randint(20, 80),
            "learning_rate": uniform(0.01, 0.3),
            "n_estimators": randint(100, 300),
            "max_depth": randint(3, 10),
            "feature_fraction": uniform(0.7, 0.3),
            "min_child_samples": randint(5, 30),
            "reg_alpha": uniform(0, 0.5),
            "reg_lambda": uniform(0, 0.5),
            "boosting_type": ["gbdt", "dart"],
        },
    ),
}

In [ ]:
# 训练所有模型
for name, (model, params) in models.items():
    trainer.train_model(model, name, param_dist=params, n_iter=1)

In [ ]:
# 比较各模型性能
trainer.compare_models()

In [ ]:
# 收集各模型在训练集和测试集上的预测概率
# train_predictions = []
model_predictions = []
model_names = []
for name, model in trainer.models.items():
    # train_pred = model.predict_proba(X_train)[:, 1]
    y_pred = model.predict_proba(X_test)[:, 1]
    model_predictions.append(y_pred)
    model_names.append(name)

    # 使用可视化类绘制各模型ROC曲线
    # ResultVisualizer.plot_roc_curve(y_test, y_pred, name)

    # 绘制特征重要性图
    # ResultVisualizer.plot_feature_importance(model, X_train.columns.tolist(), name)

In [ ]:
# 模型融合
ensemble = ModelEnsemble(trainer.models)
ensemble_preds = ensemble.ensemble_predict(X_test, method="average")

In [ ]:
# 评估融合模型
ensemble_result = ensemble.evaluate_ensemble(X_train, y_train, X_test, y_test)
ensemble_auc_train = ensemble_result["auc_train"]
ensemble_auc_test = ensemble_result["auc_test"]
ensemble_precision = ensemble_result["precision"]
ensemble_recall = ensemble_result["recall"]
ensemble_f1 = ensemble_result["f1"]

In [ ]:
# 绘制融合模型ROC曲线
ResultVisualizer.plot_roc_curve(y_test, ensemble_preds, "模型融合")

In [ ]:
# 绘制融合模型混淆矩阵
ensemble_preds_binary = (ensemble_preds > 0.5).astype(int)
ResultVisualizer.plot_confusion_matrix(y_test, ensemble_preds_binary, "模型融合")

In [ ]:
# 多模型ROC曲线对比
model_predictions.append(ensemble_preds)
model_names.append("模型融合")
ResultVisualizer.plot_multi_roc(y_test, model_predictions, model_names)

In [ ]:
headers = [
    "模型",
    "训练集-AUC",
    "测试集-AUC",
    "测试集-准确率",
    "测试集-查准率",
    "测试集-查全率",
    "测试集-F1分数",
]
table_data = []

for i, name in enumerate(model_names[:-1]):
    result = next(r for r in trainer.results if r["model"] == name)
    y_pred = (model_predictions[i] > 0.5).astype(int)

    table_data.append(
        [
            name,
            f"{result['train_auc']:.4f}",
            f"{result['test_auc']:.4f}",
            f"{accuracy_score(y_test, y_pred):.4f}",
            f"{precision_score(y_test, y_pred, zero_division=0):.4f}",
            f"{recall_score(y_test, y_pred, zero_division=0):.4f}",
            f"{f1_score(y_test, y_pred, zero_division=0):.4f}",
        ]
    )

table_data.append(
    [
        "融合模型",
        f"{ensemble_auc_train:.4f}",
        f"{ensemble_auc_test:.4f}",
        f"{accuracy_score(y_test, ensemble_preds_binary):.4f}",
        f"{ensemble_precision:.4f}",
        f"{ensemble_recall:.4f}",
        f"{ensemble_f1:.4f}",
    ]
)

print("\n" + "=" * 110)
print("所有模型性能对比".center(105))
print("=" * 110)
print(
    tabulate(
        table_data,
        headers=headers,
        tablefmt="grid",
        numalign="center",
        stralign="center",
    )
)
print("=" * 110)

In [ ]:
# 绘制性能对比柱状图
performance_df = pd.DataFrame(table_data, columns=headers)
numeric_columns = headers[2:]
performance_df[numeric_columns] = performance_df[numeric_columns].apply(pd.to_numeric)
performance_df.rename(
    columns={
        "测试集-AUC": "AUC",
        "测试集-准确率": "准确率",
        "测试集-查准率": "查准率",
        "测试集-查全率": "查全率",
        "测试集-F1分数": "F1分数",
    },
    inplace=True,
)

ResultVisualizer.plot_performance_comparison(performance_df)

### 结果分析与总结


<font color = 'red'>**用户流失预测模型实验结果深度分析**</font>

<font color = 'purple'>**🎯 一、整体模型性能对比**</font>

**📊 关键指标综合排名（测试集）**
| 模型 | AUC | 准确率 | F1 分数 | 性能亮点 |
|------|-----|--------|--------|----------|
| <font color = 'pink'>**逻辑回归**</font> | 0.7967 | 0.7310 | 0.6488 | <font color = 'yellow'>**最佳 AUC**</font> |
| <font color = 'pink'>**融合模型**</font> | 0.7913 | 0.7321 | 0.7382 | <font color = 'yellow'>**最佳 F1&查准率**</font> |
| LightGBM | 0.7915 | 0.7407 | 0.6315 | 高准确率 |
| 梯度提升树 | 0.7953 | 0.7396 | 0.6346 | 均衡表现 |
| 随机森林 | 0.7842 | 0.7303 | 0.6308 | 稳定性强 |

> 💡 **核心发现**：逻辑回归在 AUC 指标上表现最优（0.7967），而融合模型在 F1 分数（0.7382）和查准率（0.7575）上领先

<font color = 'purple'>**🔍 二、过拟合现象分析**</font>

**训练集 vs 测试集 AUC 差异**

**⚠️ 严重过拟合模型：**

- K 近邻和 XGBoost：训练集 AUC = 1.0，但测试集仅 0.75 左右，**过拟合度高达 25%**
- 随机森林：训练集 AUC = 0.9954 → 测试集 AUC = 0.7842，**泛化能力不足**

**✅ **稳健模型**：**

- 逻辑回归：训练集 AUC = 0.803 → 测试集 AUC = 0.7967，**差异仅 0.8%**
- 支持向量机：AUC = 0.7011 → AUC = 0.7104，**唯一测试集反超的模型**

<font color = 'purple'>**🌟 三、融合模型分析**</font>

🌞**性能突破点**

**🔥 **核心优势**：**

1. **查准率 0.7575** → 比第二名 LightGBM(0.6081) **提升 24.5%**
2. **F1 分数 0.7382** → 比逻辑回归(0.6488) **提升 13.8%**
3. 唯一实现**查准率>查全率**的模型

<font color = 'purple'>**📌 四、特殊模型表现分析**</font>

**❗ 模型特异性问题**

1. **朴素贝叶斯**：

   - 查全率 0.8316 → **最高召回但精度最低(0.438)**
   - 典型"宁可错杀不可放过"策略，适合高漏检成本场景

2. **XGBoost**：
   - 查全率仅 0.5544 → **最保守的预测模型**
   - 可能因过度调参导致预测偏差

<font color = 'purple'>**💎 五、核心结论**</font>

1. **最佳单一模型**：逻辑回归（AUC = 0.7967）
2. **最佳综合模型**：融合模型（F1 = 0.7382）
3. **最大潜力模型**：LightGBM（0.7915 AUC + 0.7407 准确率）
4. **最差泛化模型**：K 近邻（ΔAUC = 0.2511）
